# SplatAD / OmniRE trên PandaSet

Đọc PandaSet từ Google Drive, train trên Colab GPU và phát lại toàn cảnh qua ba camera trước. Dashboard có hai thẻ **Gốc**, **Render** và mục **Pose SMPL**.

- Mặc định dùng `STAGE="curriculum"` và R2. Thứ tự là `028` (gần) → `011` (nhiều người) → `008` (xa và bị che). Mỗi cảnh train một mô hình riêng với ba camera trước và xuất MP4 60 FPS.
- Chọn `STAGE="scene"` cho một cảnh hoặc đặt danh sách `SCENES` để tiếp tục từ cảnh mong muốn.
- Mặc định ưu tiên `MyDrive/SplatAD+OmniRe/pandaset.zip`. Có thể đặt `PANDASET_PATH` tới một ZIP khác hoặc thư mục đã giải nén chứa các cảnh `001`, `011`, …
- R1/R2 chạy quy trình trích xuất của OmniRe trên sáu camera với SMPL Neutral v1.1. Đặt `basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl` hoặc thư mục gói SMPL chính thức đã giải nén vào thư mục dự án trên Drive. Có thể xem video gốc trước bước này.

Drive lưu dữ liệu chuẩn bị trong `inputs/` và `pose-init/`, checkpoint trong `runs/`, video và metric trong `artifacts/`. Colab giải nén tạm các cảnh cần dùng vào đĩa runtime.

Các clip camera gốc được tạo ở cell xem trước từ Drive. Các clip tái tạo được xuất sau khi mỗi mô hình có checkpoint. [PandaSet](https://huggingface.co/datasets/georghess/pandaset) · [SMPL](https://smpl.is.tue.mpg.de/) · [Pose OmniRe](https://github.com/ziyc/drivestudio/blob/e59bda4fa681f829dbb1d65f0de582b0f633c450/docs/HumanPose.md)

In [ ]:
from collections import Counter
from pathlib import Path
import json
import shutil
import subprocess
import sys
import time
from zipfile import is_zipfile

from google.colab import drive, output

DRIVE = Path("/content/drive")
drive.mount(str(DRIVE))
PERSIST = DRIVE / "MyDrive/SplatAD+OmniRe"
PERSIST.mkdir(exist_ok=True)
REPO = Path("/content/SplatAD-OmniRE")
GITHUB = "https://github.com/buide03/SplatAD-OmniRE.git"
ARCHIVE = PERSIST / "SplatAD-OmniRE.zip"

progress = None

def shell(*args, cwd=None):
    if progress is not None:
        return progress.run(args, cwd)
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

if not (REPO / "versions.json").exists():
    if ARCHIVE.is_file():
        REPO.mkdir(exist_ok=True)
        shutil.unpack_archive(ARCHIVE, REPO)
    else:
        shell("git", "clone", GITHUB, REPO)
curriculum_path = REPO / "protocol/curriculum.json"
if (REPO / ".git").is_dir():
    shell("git", "pull", "--ff-only", cwd=REPO)
assert curriculum_path.is_file(), "Cần bản mã nguồn mới có lộ trình train theo độ khó."

sys.path.insert(0, str(REPO))
from research.colab import ColabProgress, checkpoint_step, latest_checkpoint, retry_environment
from research.pandaset_input import prepare_source, select_source
from research.prepare import dump_json, front_pose_summary, read_json, sha256
from research.sync_video import FRONT_CAMERAS

PANDASET_PATH = ""  # Để trống để ưu tiên pandaset.zip trong thư mục dự án rồi tự dò Drive
STAGE = "curriculum"  # curriculum | scene | smoke | pilot | confirm
DATA_PROTOCOL_VERSION = 2
RUN_ID = f"{STAGE}-front3-fullres-rgbq-v{DATA_PROTOCOL_VERSION}"
VARIANTS = ("R2",) if STAGE in ("curriculum", "scene") else ("R0", "R1", "R2")
STEPS = {"curriculum": 1200, "scene": 1200, "smoke": 200, "pilot": 1200, "confirm": 4000}[STAGE]
QUALITY_EVERY = 50 if STAGE == "smoke" else 250
CHECKPOINT_EVERY = 100
CURRICULUM = {entry["scene_id"]: entry for entry in json.loads(curriculum_path.read_text(encoding="utf-8"))["scenes"]}
PILOT = json.loads((REPO / "protocol/pilot.json").read_text(encoding="utf-8"))
if STAGE == "curriculum":
    SCENES = list(CURRICULUM)
elif STAGE in ("scene", "smoke"):
    SCENES = ["011"]
else:
    SCENES = [entry["scene_id"] for entry in PILOT["scenes"]]
TRAIN_CAMERAS = [*FRONT_CAMERAS, "left", "right", "back"] if STAGE in ("pilot", "confirm") else list(FRONT_CAMERAS)
CAMERAS = TRAIN_CAMERAS
REPLAY_CAMERAS = FRONT_CAMERAS
REPLAY_FPS = 60
REPLAY_INTERPOLATION = "smooth"  # smooth | linear
REPLAY_SCALE = 1.0  # Giữ độ phân giải gốc cho từng camera
OMNIRE_REFERENCE = "off"  # off | compare | fill
POSE_PIPELINE = "omnire-option2"
NEED_POSE = any(variant in ("R1", "R2") for variant in VARIANTS)
SMPL_FILENAME = "basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl"
SMPL_PACKAGE = PERSIST / "SMPL_python_v.1.1.0"
package_model = Path("smpl/models") / SMPL_FILENAME
model_paths = (
    PERSIST / SMPL_FILENAME,
    SMPL_PACKAGE / package_model,
    SMPL_PACKAGE / SMPL_PACKAGE.name / package_model,
    PERSIST / "SMPL_NEUTRAL.pkl",
)
SMPL_PICKLE = next((path for path in model_paths if path.is_file()), model_paths[0])
MODEL_VERSION = "smpl_neutral_v1.1"
SMPL_SOURCE_ID = sha256(SMPL_PICKLE)[:12] if NEED_POSE and SMPL_PICKLE.is_file() else None
if NEED_POSE:
    RUN_ID = f"{RUN_ID}-smpl-opacity-0.99-{POSE_PIPELINE}-{OMNIRE_REFERENCE}-{MODEL_VERSION}-{SMPL_SOURCE_ID or 'pending'}"
POSE_OUTPUT = PERSIST / "pose-init" / POSE_PIPELINE / f"{MODEL_VERSION}-{SMPL_SOURCE_ID or 'pending'}"
LOCAL_POSES = POSE_OUTPUT / "local-4d-humans"
REFERENCE_POSES = POSE_OUTPUT / "omnire-reference"
INPUTS = PERSIST / "inputs"
RUNS = PERSIST / "runs"
ARTIFACTS = PERSIST / "artifacts"
RESULTS = ARTIFACTS / RUN_ID
tag_cameras = "front3" if tuple(CAMERAS) == FRONT_CAMERAS else "surround"
assert OMNIRE_REFERENCE in ("off", "compare", "fill"), "Chọn off, compare hoặc fill."
for folder in (INPUTS, RUNS, ARTIFACTS, RESULTS):
    folder.mkdir(exist_ok=True)
progress = ColabProgress(RESULTS / "progress")

read_cameras = (*FRONT_CAMERAS, "left", "right", "back") if NEED_POSE else tuple(CAMERAS)
project_archive = PERSIST / "pandaset.zip"
dataset_path = PANDASET_PATH or (project_archive if project_archive.is_file() else None)
SOURCE, SCENES = select_source(dataset_path, [DRIVE / "MyDrive", DRIVE / "Shareddrives"], SCENES)

def require_smpl_model():
    if NEED_POSE and not SMPL_SOURCE_ID:
        raise FileNotFoundError(f"Cần SMPL Neutral v1.1. Đặt {SMPL_FILENAME} hoặc thư mục "
                                f"{SMPL_PACKAGE.name} đã giải nén trong {PERSIST} rồi chạy lại cell cấu hình.")

def raw_for(scene):
    with progress.phase(f"stage-{scene}"):
        return prepare_source(SOURCE, Path("/content/pandaset-from-drive"), [scene], read_cameras)

def publish_source(scene, raw=None, manifest=None):
    folder = ARTIFACTS / "source" / scene
    path = folder / "run.json"
    if not path.is_file() or not (folder / "front-triptych.mp4").is_file():
        shell(sys.executable, "-m", "research.source_preview", "--raw", raw or raw_for(scene),
              "--scene", scene, "--front-triptych", "--output", folder, cwd=REPO)
    if manifest:
        record = read_json(path)
        summary = front_pose_summary(read_json(manifest))
        if summary != record.get("pose_summary"):
            record["pose_summary"] = summary
            dump_json(path, record)

print(f"PandaSet: {SOURCE}")
print(f"Tiến độ và log trên Drive · {progress.folder}")
if SMPL_SOURCE_ID:
    print(f"SMPL Neutral · {MODEL_VERSION} · {SMPL_PICKLE.name}")
print("Thứ tự train", " → ".join(SCENES))

## Xem video Gốc và Pose SMPL

Cell này tạo video **Gốc** đồng bộ ba camera trước của cảnh đầu tiên và mở dashboard. Video ở thẻ **Render** xuất hiện sau khi có checkpoint. Mục **Pose SMPL** tự cập nhật khi chạy R1 hoặc R2.

In [ ]:
if shutil.which("ffmpeg") is None:
    shell("apt-get", "-qq", "update")
    shell("apt-get", "-qq", "install", "-y", "ffmpeg")

publish_source(SCENES[0])

if "dashboard" in globals() and dashboard.poll() is None:
    dashboard.terminate()
    dashboard.wait(timeout=5)
dashboard = subprocess.Popen([sys.executable, "-m", "research.dashboard",
                              "--artifacts", str(ARTIFACTS), "--port", "8765"], cwd=REPO)
time.sleep(1)
assert dashboard.poll() is None, "Không khởi động được dashboard."
output.serve_kernel_port_as_iframe(8765, height=1000)

## Môi trường train

Chọn GPU trong Colab trước khi chạy cell này. Sau khi runtime bị ngắt, kết nối lại và chạy lại notebook với cùng cấu hình lượt train. Bootstrap dùng Python 3.12, PyTorch CUDA 12.8 hoặc 12.6 và fork GSplat theo commit đã pin. Sau khi nạp checkpoint, repo render ba camera theo cùng mốc thời gian rồi ghép thành một clip.

In [ ]:
def bootstrap(profile, repair=False):
    options = ("--repair",) if repair else ()
    phase = f"repair-{profile}" if repair else f"environment-{profile}"
    with progress.phase(phase):
        shell("bash", "scripts/bootstrap-colab.sh", profile, *options, cwd=REPO)

def research(module, *args):
    def launch():
        shell("bash", "-c", 'source "$1"; shift; exec python -m "$@"',
              "splatad", REPO / ".deps/train-env/activate-research", f"research.{module}", *args, cwd=REPO)

    if module == "train":
        return launch()
    return retry_environment(launch, lambda: bootstrap("train", repair=True))

with progress.phase("check-runtime"):
    require_smpl_model()
    shell("nvidia-smi")
bootstrap("train")

## Pose khởi tạo và manifest

OmniRe Option 2 chiếu cuboid người lên ảnh, chạy 4D-Humans trên sáu camera rồi chạy Tracklet Matching và Pose Completion. Mặc định `OMNIRE_REFERENCE="off"` dùng track do quy trình này tạo. Chọn `"compare"` để đo chênh lệch với pose công bố, hoặc `"fill"` cho lần chạy thử kết hợp hai nguồn. Kết quả lưu trong `MyDrive/SplatAD+OmniRe/pose-init/omnire-option2/`.

In [ ]:
BODY = POSE_OUTPUT / "neutral.npz"

def convert_body():
    shell("bash", "-c", 'source "$1"; shift; exec python -m research.prepare "$@"', "splatad",
          REPO / ".deps/smpl-convert/activate-research", "prepare-smpl",
          "--input", SMPL_PICKLE, "--output", BODY, "--model-version", MODEL_VERSION, cwd=REPO)

if NEED_POSE:
    with progress.phase("prepare-smpl-model"):
        require_smpl_model()
        if not is_zipfile(BODY):
            bootstrap("convert")
            retry_environment(convert_body, lambda: bootstrap("convert", repair=True))

pose_env_ready = False

def pose_folder(scene):
    return POSE_OUTPUT / "scenes" / scene / f"export-v{DATA_PROTOCOL_VERSION}"

def tracks_for(scene):
    return pose_folder(scene) / OMNIRE_REFERENCE / "tracks.json"

def manifest_for(scene):
    if NEED_POSE:
        return tracks_for(scene).with_name(f"manifest-{tag_cameras}.json")
    return INPUTS / f"{scene}-{tag_cameras}-rigid-manifest-v{DATA_PROTOCOL_VERSION}.json"

def prepare_pose(scene, raw):
    global pose_env_ready
    final = tracks_for(scene)
    report = final.with_name("comparison.json")
    if final.is_file() and report.is_file():
        return
    primary = pose_folder(scene) / "tracks-4d-humans.json"
    if not primary.is_file():
        local = LOCAL_POSES / scene
        if not all((local / file).is_file() for file in ("humanpose/smpl.pkl", "instances/instances_info.json")):
            if not pose_env_ready:
                bootstrap("pose")
                pose_env_ready = True
            retry_environment(
                lambda: shell("bash", "scripts/omnire-pose.sh", raw, LOCAL_POSES, scene, SMPL_PICKLE, cwd=REPO),
                lambda: bootstrap("pose", repair=True),
            )
        research("prepare", "export-tracks", "--raw", raw, "--processed", LOCAL_POSES,
                 "--body", BODY, "--scene", scene, "--output", primary)
    reference_args = []
    if OMNIRE_REFERENCE != "off":
        reference = pose_folder(scene) / "tracks-omnire-reference.json"
        source_pose = REFERENCE_POSES / scene / "humanpose/smpl.pkl"
        if not reference.is_file():
            if not source_pose.is_file():
                research("omnire", "--raw", raw, "--processed", REFERENCE_POSES, "--allow-missing",
                         "--archive", INPUTS / "pandaset_processed_humanpose.zip", "--scenes", scene)
            if source_pose.is_file():
                research("prepare", "export-tracks", "--raw", raw, "--processed", REFERENCE_POSES,
                         "--body", BODY, "--scene", scene, "--output", reference)
        if reference.is_file():
            reference_args = ["--reference", reference]
    research("pose_init", "--primary", primary, *reference_args, "--body", BODY,
             "--mode", OMNIRE_REFERENCE, "--output", final, "--report", report)
    print(f"Pose đã lưu trên Drive · {final.parent}")

def check_curriculum(scene, spec):
    stage = CURRICULUM[scene]
    selected = set(spec["selected_uuids"])
    moving = [row for row in spec["observations"]
              if row["uuid"] in selected and row["smpl_valid"] and not row["stationary"]]
    near = {row["uuid"] for row in moving if row["bin"] == "near" and row["usable"]}
    per_frame = Counter((row["camera"], row["frame"]) for row in moving if row["usable"])
    simultaneous = max(per_frame.values(), default=0)
    far_occluded = {row["uuid"] for row in moving if row["bin"] == "far"
                    and row["limbs_visible"] is False and row["pose_monitor"]}
    print(f"{scene} · {stage['label']} · {len(near)} track gần · đỉnh {simultaneous} người cùng khung · "
          f"{len(far_occluded)} track xa có mốc bị che để monitor")
    qualifies = {"near": bool(near), "multiple": simultaneous >= 3, "far_occluded": bool(far_occluded)}
    if not qualifies[stage["focus"]]:
        raise ValueError(f"Manifest {scene} thiếu quan sát SMPL cho trọng tâm {stage['label']}")

def prepare_scene(scene, raw):
    tracks, manifest = tracks_for(scene), manifest_for(scene)
    if NEED_POSE:
        prepare_pose(scene, raw)
    if not manifest.exists():
        review = INPUTS / f"{scene}-review.json"
        if not review.is_file():
            review = REPO / "protocol" / f"{scene}-review.json"
        research("prepare", "inventory", "--raw", raw, "--scene", scene,
                 "--cameras", *CAMERAS, "--output", manifest,
                 *(["--tracks", tracks] if NEED_POSE else []),
                 *(["--review", review] if review.is_file() else []))
    if NEED_POSE:
        spec = json.loads(manifest.read_text(encoding="utf-8"))
        print(scene, len(spec["selected_uuids"]), "UUID định tuyến SMPL")
        assert spec["selected_uuids"], f"Cần track SMPL hợp lệ cho cảnh {scene}"
        if STAGE == "curriculum" and scene in CURRICULUM:
            check_curriculum(scene, spec)
        if STAGE in ("pilot", "confirm"):
            simultaneous = {}
            for row in spec["observations"]:
                if row["usable"] and row["smpl_valid"] and row["uuid"] in spec["selected_uuids"]:
                    simultaneous.setdefault((row["camera"], row["frame"]), set()).add(row["uuid"])
            assert max(map(len, simultaneous.values()), default=0) >= 2, f"Cảnh {scene} cần hai UUID SMPL cùng xuất hiện"

## Train và xuất toàn cảnh theo từng checkpoint

Mỗi cảnh chạy SplatAD gốc trước, rồi train các cấu hình nghiên cứu trên cùng bộ ảnh và crop cố định. Tất cả các lượt train RGB ở độ phân giải gốc từ bước 0. Dashboard cập nhật PSNR và SSIM của toàn cảnh, xe và người từ checkpoint đầu, theo mốc và khi hoàn tất. Notebook lưu checkpoint mỗi 100 bước và tiếp tục từ checkpoint hợp lệ mới nhất của từng cảnh. Log và trạng thái từng giai đoạn nằm trong thư mục progress trên Drive. Các lượt có R1/R2 dùng mã run mới để đo lại từ bước 0 với opacity SMPL là 0.99.

In [ ]:
def run_folder(scene, variant):
    return RUNS / f"{scene}-{variant}" / "splatad-smpl" / RUN_ID

def checkpoint_for(scene, variant):
    return latest_checkpoint(run_folder(scene, variant))

def replay_args(scene, variant, zero_velocity=False):
    checkpoint = checkpoint_for(scene, variant)
    suffix = "-velocity-off" if zero_velocity else ""
    folder = RESULTS / f"{scene}-{variant}{suffix}-{checkpoint.stem}-{REPLAY_FPS}fps-{REPLAY_SCALE:g}x-{REPLAY_INTERPOLATION}"
    shared = ["--config", run_folder(scene, variant) / "config.yml", "--checkpoint", checkpoint,
              "--manifest", manifest_for(scene), "--output", folder]
    if zero_velocity:
        shared.append("--zero-smpl-velocity")
    return folder, shared

def replay_ready(folder):
    artifact = folder / "run.json"
    if not artifact.is_file():
        return False
    record = read_json(artifact)
    triptych = record.get("front_triptych") or {}
    files = [entry["file"] for entry in record.get("videos", [])]
    return (triptych.get("cameras") == list(REPLAY_CAMERAS) and len(files) == len(REPLAY_CAMERAS)
            and all((folder / file).is_file() and (folder / file).stat().st_size
                    for file in [*files, triptych["file"]]))

def render_scene(scene, variant, zero_velocity=False):
    folder, shared = replay_args(scene, variant, zero_velocity)
    if replay_ready(folder):
        print(f"Replay đã có: {folder}")
        return
    suffix = "-velocity-off" if zero_velocity else ""
    with progress.phase(f"render-{scene}-{variant}{suffix}"):
        research("replay", *shared, "--cameras", *REPLAY_CAMERAS, "--fps", REPLAY_FPS,
                 "--scale", REPLAY_SCALE, "--interpolation", REPLAY_INTERPOLATION)

def train_scene(scene, variant, raw, quality_set):
    folder = run_folder(scene, variant)
    checkpoint = latest_checkpoint(folder, include_initial=True)
    done = checkpoint_step(checkpoint)
    if done >= STEPS:
        print(f"{scene} {variant} đã hoàn thành {done} bước")
        return
    live = ["--quality-set", quality_set, "--quality-dir", RESULTS / f"{scene}-{variant}-quality",
            "--quality-every", QUALITY_EVERY]
    if variant in ("R1", "R2"):
        live.extend(["--pose-monitor-dir", RESULTS / f"{scene}-{variant}-pose",
                     "--pose-monitor-every", QUALITY_EVERY])
    if checkpoint:
        print(f"{scene} {variant} tiếp tục từ {checkpoint.name} sau {done} bước", flush=True)
        research("train", "--resume", folder / "config.yml", "--checkpoint", checkpoint, "--raw", raw,
                 "--steps", STEPS - done, "--checkpoint-every", CHECKPOINT_EVERY, *live)
        return
    pose = ["--tracks", tracks_for(scene), "--body", BODY] if variant in ("R1", "R2") else []
    manifest = ["--manifest", manifest_for(scene)] if variant != "SplatAD" else []
    research("train", "--raw", raw, "--scene", scene, "--variant", variant, *pose, *manifest,
             "--cameras", *CAMERAS, "--output", RUNS, "--run-id", RUN_ID, "--steps", STEPS,
             "--downscales", 0,
             "--checkpoint-every", CHECKPOINT_EVERY,
             "--monitor-every", 0 if variant == "SplatAD" else (200 if STAGE == "smoke" else 500), *live)

def train_with_recovery(scene, variant, raw, quality_set):
    with progress.phase(f"train-{scene}-{variant}"):
        retry_environment(lambda: train_scene(scene, variant, raw, quality_set),
                          lambda: bootstrap("train", repair=True))
        if checkpoint_step(checkpoint_for(scene, variant)) < STEPS:
            raise RuntimeError(f"{scene} {variant} chưa có checkpoint đạt {STEPS} bước.")

for stage_index, scene in enumerate(SCENES, start=1):
    focus = CURRICULUM.get(scene, {}).get("label", "") if STAGE == "curriculum" else ""
    print(f"[{stage_index}/{len(SCENES)}] Cảnh {scene} · {focus or STAGE}", flush=True)
    if (all(checkpoint_step(checkpoint_for(scene, variant)) >= STEPS for variant in ("SplatAD", *VARIANTS))
            and all(replay_ready(replay_args(scene, variant)[0]) for variant in VARIANTS)):
        print(f"Cảnh {scene} đã có checkpoint và video, bỏ qua bước train và render.")
        continue
    raw = raw_for(scene)
    quality_set = INPUTS / f"{scene}-{tag_cameras}-rgbq-v{DATA_PROTOCOL_VERSION}.json"
    with progress.phase(f"prepare-{scene}"):
        prepare_scene(scene, raw)
        publish_source(scene, raw, manifest_for(scene))
        if not quality_set.is_file():
            research("quality", "--raw", raw, "--scene", scene, "--cameras", *CAMERAS,
                     "--manifest", manifest_for(scene), "--output", quality_set)
    train_with_recovery(scene, "SplatAD", raw, quality_set)
    for variant in VARIANTS:
        train_with_recovery(scene, variant, raw, quality_set)
        render_scene(scene, variant)

## Metric và đối chứng

Cell này bổ sung metric cho các replay đã xuất. Đánh giá dùng ảnh và tia tại mốc cảm biến gốc. Đặt `CHECK_VELOCITY=True` để render thêm đối chứng R2/vận tốc 0. Đặt `RUN_GRADIENT_PROBES=True` để chạy probe RGB/LiDAR từ checkpoint sau khi xuất video.

In [ ]:
PROBES_ONLY = STAGE != "confirm"
CHECK_VELOCITY = STAGE in ("smoke", "pilot", "confirm")
RUN_GRADIENT_PROBES = STAGE in ("smoke", "pilot", "confirm")
folders = []
staged = set()
metric_scope = "probes" if PROBES_ONLY else "full"

def stage_for_evaluation(scene):
    if scene not in staged:
        raw_for(scene)
        staged.add(scene)

def run_pending_check(scene, phase, output, module, *args):
    if output.is_file() and progress.records.get(phase, {}).get("status") == "complete":
        return
    stage_for_evaluation(scene)
    with progress.phase(phase):
        research(module, *args)

for scene in SCENES:
    for variant in VARIANTS:
        velocity_modes = (False, True) if CHECK_VELOCITY and variant == "R2" else (False,)
        for zero_velocity in velocity_modes:
            folder, shared = replay_args(scene, variant, zero_velocity)
            if not replay_ready(folder):
                stage_for_evaluation(scene)
                render_scene(scene, variant, zero_velocity=zero_velocity)
            run_pending_check(scene, f"evaluate-{metric_scope}-{folder.name}", folder / "metrics.json",
                              "evaluate", *shared, *(["--probes"] if PROBES_ONLY else []))
            folders.append(folder)
if len(VARIANTS) > 1 or (CHECK_VELOCITY and "R2" in VARIANTS):
    with progress.phase("compare"):
        research("compare", "--runs", *folders, "--output", RESULTS / "comparison")

if RUN_GRADIENT_PROBES:
    for scene in SCENES:
        for variant in VARIANTS:
            if variant in ("R0", "R2"):
                checkpoint = checkpoint_for(scene, variant)
                # Include the trained step in the filename and phase name so different checkpoints can be inspected.
                phase = f"{scene}-{variant}-{checkpoint.stem}-probe"
                output = RESULTS / f"{phase}.json"
                run_pending_check(scene, phase, output, "probe", "--config", run_folder(scene, variant) / "config.yml",
                                  "--checkpoint", checkpoint, "--manifest", manifest_for(scene), "--output", output)

## Dashboard sau khi render

Chọn cảnh rồi chuyển giữa **Gốc** và **Render**. Bấm tên camera để xem riêng và **Cả ba** để trở lại. Trong **Pose SMPL**, nút **Tới khung hình** mở đúng camera và mốc đang so khớp.

In [ ]:
if "dashboard" not in globals() or dashboard.poll() is not None:
    dashboard = subprocess.Popen([sys.executable, "-m", "research.dashboard",
                                  "--artifacts", str(ARTIFACTS), "--port", "8765"], cwd=REPO)
    time.sleep(1)
output.serve_kernel_port_as_iframe(8765, height=1000)